In [2]:
import matplotlib.pyplot as plt
from qiskit.visualization import plot_histogram
import numpy as np
from qiskit import QuantumCircuit, QuantumRegister
import time
from functools import partial
from qiskit.result import Counts
from collections import Counter
from qiskit.circuit import ParameterVector
from qiskit.circuit.library import EfficientSU2, ExcitationPreserving
from qiskit.compiler import transpile
from qiskit.quantum_info import SparsePauliOp, Operator
from qiskit.primitives import StatevectorEstimator
from qiskit.transpiler import CouplingMap
%matplotlib inline
import openfermion as of

from iqm import qiskit_iqm
from iqm.qiskit_iqm.fake_backends import fake_aphrodite
from iqm.qiskit_iqm.iqm_transpilation import optimize_single_qubit_gates, IQMOptimizeSingleQubitGates
from qiskit_ibm_runtime import Estimator, Session, Options, Batch

from typing import List
from scipy.optimize import minimize, dual_annealing, basinhopping, cobyla, BFGS
from qiskit_algorithms.optimizers import SPSA
from qiskit.result import CorrelatedReadoutMitigator
import scipy.linalg as la

from iqm.qiskit_iqm import IQMProvider
import os

In [ ]:
# Setting backend 
fake_backend = fake_aphrodite.IQMFakeAphrodite()
target = fake_backend.target

#Q50_CORTEX_URL = os.getenv('Q50_CORTEX_URL')
#Q50_provider = IQMProvider(Q50_CORTEX_URL)
#fake_backend = Q50_provider.get_backend()
# target = fake_backend.target

# Hubbard parameters
N_x = 1            # lattice sites in x-direction
N_y = 3            # lattice sites in y-direction
N = N_x * N_y
U = 50             # repulsive on-site interaction
t = 1.0            # kinetic energy

layout = [4,5,6,10,11,12]   # choice of qubits to use

# Hubbard Hamiltonian
hubbard_hamiltonian = of.fermi_hubbard(N_x, N_y, tunneling=t, coulomb=U, chemical_potential=U/2, periodic=False)
print()
print("Hubbard Hamiltonian:")
print(hubbard_hamiltonian)

# Jordan-Wigner transformation
pauli_hubbard_hamiltonian = of.transforms.jordan_wigner(hubbard_hamiltonian)
print()
print("Hubbard Hamiltonian in Pauli basis:")
print(pauli_hubbard_hamiltonian)
print("List of terms")
print(pauli_hubbard_hamiltonian.terms)

# Converting Hamiltonian from openfermion to Qiskit SparsePauliOp
def openfermion_to_sparsepauliop(N, qubit_op):
    labels = []
    coeffs = []

    # Get number of qubits by checking all indices in terms
    max_index = 0
    for term in qubit_op.terms:
        for qubit, _ in term:
            if qubit > max_index:
                max_index = qubit
    n_qubits = max_index + 1

    for term, coeff in qubit_op.terms.items():
        label = ['I'] * 2*N
        for qubit, pauli in term:
            label[qubit] = pauli
        labels.append(''.join(reversed(label)))
        coeffs.append(complex(coeff))

    return SparsePauliOp(labels, coeffs=np.array(coeffs))

# Final Hubbard Hamiltonian
hamiltonian1 = openfermion_to_sparsepauliop(N, pauli_hubbard_hamiltonian)
print(hamiltonian1)

# Efficient SU2 ansatz
ansatz_SU2 = EfficientSU2(2*N, reps=1, su2_gates="r", entanglement="circular")

# Half-filled initial state
def half_filled_initial_state(n_qubits, n_particles):
    qc = QuantumCircuit(n_qubits)
    for i in range(n_particles):
        qc.x(i)
    return qc

initial_state = half_filled_initial_state(2*N, N)

# Physically inspired adapted SU(3) fermion ansatz
def givens_rotation(circuit, i, j, theta):
    # Applies a number-conserving Givens rotation between qubits i and j
    circuit.ry(-np.pi / 2, j)
    circuit.cx(i, j)
    circuit.ry(theta, j)
    circuit.cx(i, j)
    circuit.ry(np.pi / 2, j)

def fermionic(filling, N, layers):
    """
        filling: number of occupied modes for spin up/down as a list, for example [2, 2] is 2 spin up electrons, 2 spin down electrons
        N: number of spatial sites
        layers: number of ansatz layers
    """
    nqubits = 2 * N
    circuit = QuantumCircuit(nqubits)
    theta = ParameterVector('r', 5 * layers * (nqubits - 1))
    k = 0

    # Initial state preparation based on filling
    for color, n_particles in enumerate(filling):
        count = 0
        for site in range(N):
            if count < n_particles:
                qubit = color * N + site
                circuit.x(qubit)
                count += 1

    # Variational layers of Givens Rotations
    for layer in range(layers):
        # Even layer
        for i in range(0, nqubits - 1, 2):
            givens_rotation(circuit, i, i + 1, theta[k])
            k += 1
        # Odd layer
        for i in range(1, nqubits - 1, 2):
            givens_rotation(circuit, i, i + 1, theta[k])
            k += 1
        # Spin-up-spin-down entanglers 
        for i in range(N):
            q_up = i
            q_down = i + N
            givens_rotation(circuit, q_up, q_down, theta[k])
            k += 1
    return circuit

ansatz_fermionic = fermionic([1, 1], N, 1)

# Hamiltonian Variational Ansatz (custom HVA ansatz)
def givens_rotation(circuit, i, j, theta):
    """Applies a Givens rotation (number-conserving) between qubits i and j."""
    circuit.ry(-np.pi / 2, j)
    circuit.cx(i, j)
    circuit.ry(theta, j)
    circuit.cx(i, j)
    circuit.ry(np.pi / 2, j)

def hva_ansatz(N, layers):
    """
        N: number of spatial sites
        layers: number of HVA circuit layers
    """
    n_qubits = 2 * N                                           # 2N qubits for N electrons
    circuit = QuantumCircuit(n_qubits)
    theta_t = ParameterVector('θ_t', layers * (n_qubits - 1))  # hopping terms
    theta_u = ParameterVector('θ_u', layers * N)               # interaction terms
    k_t = 0
    k_u = 0

    # Initial state
    circuit.x(0)
    circuit.x(N)

    # HVA layers
    for l in range(layers):
        # Hopping terms
        for i in range(0, n_qubits - 1):
            givens_rotation(circuit, i, i + 1, theta_t[k_t])
            k_t += 1

        # On-site interaction terms
        for site in range(N):
            q_down = site
            q_up = site + N
            circuit.rzz(theta_u[k_u], q_down, q_up)
            k_u += 1

    return circuit

ansatz_HVA_custom = hva_ansatz(N, 1)

# Setting ansatz and estimator
ansatz1 = initial_state.compose(ansatz_SU2)
print("Ansatz qubit count:", ansatz1.num_qubits)
session = Session(backend=fake_backend)
estimator = Estimator(mode=session)
estimator.options.default_shots = 1000

# Compiling ansatz to backend
transpiled_circuit = transpile(ansatz1, target=fake_backend.target, layout_method="trivial", routing_method="sabre", optimization_level=3)
ansatz2 = optimize_single_qubit_gates(transpiled_circuit)

# Removing unused qubits
def remove_unused_qubits(circuit: QuantumCircuit) -> QuantumCircuit:
    # Identify used qubits
    used_qubits = set()
    for instr, qargs, _ in circuit.data:
        for qubit in qargs:
            used_qubits.add(qubit)

    # Map old qubits to new indices
    qubit_map = {qubit: idx for idx, qubit in enumerate(sorted(used_qubits, key=lambda q: circuit.qubits.index(q)))}
    new_circ = QuantumCircuit(len(used_qubits))

    # Rebuild the circuit
    for instr, qargs, cargs in circuit.data:
        new_qargs = [new_circ.qubits[qubit_map[q]] for q in qargs]
        new_circ.append(instr, new_qargs, cargs)

    return new_circ

ansatz = remove_unused_qubits(ansatz2)

# Printing number of qubits and total number of gates of compiled circuit
print("Compiled circuit qubit count:", ansatz.num_qubits)
print("Compiled circuit gate count:", len(ansatz))
print("Two-qubit gates:", ansatz.count_ops())

# Matching Hamiltonian dimension to circuit dimension
def pad_hamiltonian(ham: SparsePauliOp, total_qubits: int) -> SparsePauliOp:
    current_qubits = ham.num_qubits
    if current_qubits == total_qubits:
        return ham  # no change needed
    elif current_qubits > total_qubits:
        raise ValueError("Hamiltonian has more qubits than the circuit.")

    # Padding with identities on left side (least significant qubits)
    n_pad = total_qubits - current_qubits
    padding = SparsePauliOp.from_list([('I' * n_pad, 1.0)])
    padded_ham = ham.tensor(padding)  # Pads on left
    return padded_ham

hamiltonian = pad_hamiltonian(hamiltonian1, ansatz.num_qubits)

class Learning:
	def __init__(self, ansatz, backend, hamiltonian, estimator, optimizer: List[float]) -> None:
		self.optimizer = optimizer
		self.backend = backend
		self.seed = None
		self.hamiltonian = hamiltonian
		self.estimator = estimator
		self.ansatz = ansatz
		self.energy = None
		self.num_params = ansatz.num_parameters
		self.n = ansatz.num_qubits
		self.min_kwargs = None
		self.x0 = None
		self.bounds = None
		self.params = None
		self.cost = None
		self.p = None
		self.iter = None
		self.nfev = None
		self.result = None
		self.shots = None
		self.cost_function = None
        
		self.costs_dictionary = {
            "params": None,
            "iterations": 0,
            "costs": [],
        }

	def run(self,
	        min_kwargs: dict | None = None,
	        x0: List[float] | None = None,
            params: List[float] | None = None,
	        shots: int | None = None,
	        seed: int | None = None
	        ) -> dict:
		optimizer = self.optimizer
		self.num_params = self.ansatz.num_parameters
		self.x0 = np.random.normal(0, 0.01, self.ansatz.num_parameters) 
		self.bounds = [(-np.pi, np.pi)] * self.num_params
		self.min_kwargs = min_kwargs if min_kwargs else dict()
		self.cost_function = self.energy_cost_function

		if optimizer == 'dual_annealing':
			self.optimizer = dual_annealing
			self.result = dual_annealing(func=self.cost_function, bounds=self.bounds, callback=self.callback)
		elif optimizer == 'basinhopping':
			self.optimizer = basinhopping
			self.result = basinhopping(func=self.cost_function, x0=self.x0)
		elif optimizer == 'cobyla':
			self.result = minimize(self.cost_function, self.x0, method='COBYLA', tol=1e-5, options={"maxiter":5000})#, "rhobeg": 0.5})
		elif optimizer == 'BFGS':
			self.result = minimize(self.cost_function, self.x0, method='BFGS', tol=1e-5, options={"maxiter": 5000})
		elif optimizer == 'SPSA':
			# SPSA with possible decaying perturbation size and learning rate 
			def two_phase_lr(a0 = 0.0000002, a1=0.02, a2=0.02, switch_iter1=3, switch_iter2=70):
				def generator():
					for k in range(100):
						if k < switch_iter1:
							yield a0
						elif k > switch_iter1 and k < switch_iter2:
							yield a1
						else:
							yield a2
				return generator

			def two_phase_pert(c0 = 0.0000001, c1=0.3, c2=0.3, switch_iter1=3, switch_iter2=70):
				def generator():
					for k in range(100):
						if k < switch_iter1:
							yield c0
						elif k > switch_iter1 and k < switch_iter2:
							yield c1
						else:
							yield c2
				return generator
			spsa = SPSA(
				maxiter=75,
				learning_rate=two_phase_lr(a0 = 0.00000002, a1=0.02, a2=0.02, switch_iter1 = 3, switch_iter2=70),
				perturbation=two_phase_pert(c0 = 0.00000001, c1=0.3, c2=0.3, switch_iter1 = 3, switch_iter2=70))
			self.result = spsa.minimize(self.cost_function, self.x0)
		elif optimizer == 'SPSA2':
			spsa = SPSA(maxiter=100)
			self.result = spsa.minimize(self.cost_function, self.x0)

			self.params = self.result.x
			return dict(
				n=self.n,
				iter=self.iter,
				nfev=self.nfev,
				cost=self.cost,
				params=self.params,
				ansatz=self.ansatz,
				optimizer=self.optimizer.__class__.__name__,
				min_kwargs=self.min_kwargs,
				shots=self.shots)

	# Defining count filtering function
	def filter_counts_by_particle_number(counts: Counts, target_particles: int) -> Counts:
		filtered_counts = {
			bitstr: count for bitstr, count in counts.items()
			if bitstr.count('1') == target_particles
		}
		total = sum(filtered_counts.values())
		if total == 0:
			raise ValueError("No bitstrings matched the target particle number.")
		original_total = sum(filtered_counts.values())
		probabilities = {bitstr: count/original_total for bitstr, count in filtered_counts.items()}
		
		scaled_counts = {}
		remainders = []
		
		for bitstr, prob in probabilities.items():
			exact = prob * 1000
			integer = int(exact)
			scaled_counts[bitstr] = integer
			remainders.append((bitstr, exact - integer))
		
		remaining = 1000 - sum(scaled_counts.values())
		for bitstr, remainder in sorted(remainders, key=lambda x: -x[1])[:remaining]:
			scaled_counts[bitstr] += 1
		
		return scaled_counts
	
	def calibration_circuits(num_qubits):
		circuits = []
		for i in range(2**num_qubits):
			state = format(i, f'0{num_qubits}b')
			qc = QuantumCircuit(num_qubits, num_qubits)
                
			for j, bit in enumerate(reversed(state)):
				if bit == '1':
					qc.x(j)
                
			qc.measure(range(num_qubits), range(num_qubits))
			circuits.append(qc)
            
		return circuits

	def mitigation_matrix(calibration_results, num_qubits):
		num_states = 2**num_qubits
		M = np.zeros((num_states, num_states))
            
		for state_index in range(num_states):
			counts = calibration_results.get_counts(state_index)
			total_shots = sum(counts.values())
                
			for measured_state, count in counts.items():
				measured_state_index = int(measured_state, 2)
				M[measured_state_index, state_index] = count / total_shots
            
		return M

	def mitigate_counts(noisy_counts, Minv, num_qubits, shots):
		counts_vector = np.zeros(2**num_qubits)
		for state, count in noisy_counts.items():
			index = int(state, 2)
			counts_vector[index] = count
            
		mitigated_vector = np.dot(Minv, counts_vector)
            
		mitigated_vector = np.maximum(mitigated_vector, 0)
		mitigated_vector = mitigated_vector * shots / np.sum(mitigated_vector)
            
		mitigated_counts = {}
		for i in range(len(mitigated_vector)):
			state = format(i, f'0{num_qubits}b')
			mitigated_counts[state] = round(mitigated_vector[i])
            
		return mitigated_counts

	backend = fake_aphrodite.IQMFakeAphrodite()
	num_qubits = 6
	shots = 10000

	# Generating and running calibration circuits
	calibration_circuits_1 = calibration_circuits(num_qubits)
	calibration_circuits = transpile(calibration_circuits_1, backend=backend, initial_layout=layout, optimization_level=3) 
	calibration_job = backend.run(calibration_circuits, shots=shots)
	calibration_results = calibration_job.result()
	print(calibration_results.get_counts())

	# Building mitigation matrix
	M = mitigation_matrix(calibration_results, num_qubits)
	print("Calibration matrix M:")
	print(M)
    
	# Calculating inverse matrix
	Minv = la.inv(M)
	print("Inverse matrix Minv:")
	print(Minv)

	def energy_cost_function(self, x: List[float]) -> float:	
		self.params = x
		
		# Compiling with Qiskit to backend and binding parameters
		values = self.params
		bound_circuit = self.ansatz.assign_parameters(values)
		bound_circuit.measure_all()
		transpiled_job = transpile(bound_circuit, backend=self.backend, initial_layout=layout, optimization_level=3)
    
		mitigator = CorrelatedReadoutMitigator(self.M, qubits=layout)
        
		def is_diagonal(pauli_str):
			return all(p in {'I', 'Z'} for p in pauli_str)

		diagonal_terms = []
		diagonal_coeffs = []
		non_diagonal_terms = []
		non_diagonal_coeffs = []

		for p, c in zip(self.hamiltonian.paulis.to_labels(), self.hamiltonian.coeffs):
			if is_diagonal(p):
				diagonal_terms.append(p)
				diagonal_coeffs.append(c)
			else:
				non_diagonal_terms.append(p)
				non_diagonal_coeffs.append(c)

		diagonal_op = SparsePauliOp(diagonal_terms, diagonal_coeffs)
		non_diagonal_op = SparsePauliOp(non_diagonal_terms, non_diagonal_coeffs)

		total_mitigated_energy = 0.0
		mitigated_energy = 0.0
		total_expval = 0.0
		energies_diag = []
    
		circuits_batch = [transpiled_job] * 2
		batch_job = self.backend.run(circuits_batch, shots=10000)  # Submitting in batch
		batch_results = batch_job.result()

		energies_diag = []
        # Noise averaging
		for i in range(2):
			counts = batch_results.get_counts(i)
			energy_this_run = 0.0
			for pauli_str, coeff in zip(diagonal_terms, diagonal_coeffs):
				diag_vector = np.real(Operator.from_label(pauli_str).to_matrix().diagonal())
				expval, _ = mitigator.expectation_value(counts, diag_vector)
				energy_this_run += coeff * expval
			energies_diag.append(energy_this_run)

		mitigated_energy = np.mean(energies_diag)
		total_mitigated_energy += mitigated_energy
		self.total_energy = np.real(total_mitigated_energy)
		self.cost = self.total_energy
        
		self.costs_dictionary["iterations"] += 1
		self.costs_dictionary["parameters"] = self.params
		self.costs_dictionary["costs"].append(self.total_energy)
		print(f"Iterations: {self.costs_dictionary['iterations']} | Current cost: {self.total_energy}")
		return self.total_energy

print("Final circuit qubit count:", ansatz.num_qubits)
print("Hamiltonian qubit count:", hamiltonian.num_qubits)

"""Choosing optimizer (options include: "BFGS", "SPSA", "basinhopping", "dual_annealing", and "cobyla")
SPSA is recommended for running on real QPU (most tolerant against noise and uses only 2 circuit runs per iteration)"""
optimizer = "SPSA"
learning1 = Learning(ansatz, fake_backend, hamiltonian, estimator, optimizer)
start = time.time()
result = learning1.run()
session.close()
end = time.time()
runtime = end - start
print("Runtime:", runtime)
print(f"VQE runtime: {runtime:.2f} seconds")
print(f'Calculated cost: {learning1.total_energy}')
print(f'Calculated parameters: {learning1.params}')

H_matrix = Operator(hamiltonian).data
eigenvalues = np.linalg.eigvalsh(H_matrix)
print()
print("Min eigenvalue (exact ground state):", np.min(eigenvalues))
print("Energy error:", abs(np.min(eigenvalues))-abs(learning1.total_energy))
print("Energy per site:", np.min(eigenvalues)/N)

print("Transpiled circuit length:", len(ansatz2))
energies1 = learning1.costs_dictionary["costs"]
num_iters = len(learning1.costs_dictionary["costs"])

# --------- Plotting energy convergence ----------- #

f1 = plt.figure(figsize=(5,5))
ax1 = f1.add_subplot(1, 1, 1)
ax1.plot(energies1, color = 'blue', linewidth=1.5, label="VTT Q50")
plt.hlines(np.min(eigenvalues), xmin=0, xmax=201, label='Exact', color='black', linestyles='dashed', linewidth=1.5)
plt.xlim(0, 150)
plt.xlabel("Iteration")
plt.ylabel("Energy")
plt.title("Hubbard model VQE on Q50: Energy Convergence with REM")
plt.grid(True)
plt.tight_layout()
plt.legend()
f1.savefig("convergence_plot")
plt.show()


Hubbard Hamiltonian:
-25.0 [0^ 0] +
50.0 [0^ 0 1^ 1] +
-1.0 [0^ 2] +
-25.0 [1^ 1] +
-1.0 [1^ 3] +
-1.0 [2^ 0] +
-25.0 [2^ 2] +
50.0 [2^ 2 3^ 3] +
-1.0 [2^ 4] +
-1.0 [3^ 1] +
-25.0 [3^ 3] +
-1.0 [3^ 5] +
-1.0 [4^ 2] +
-25.0 [4^ 4] +
50.0 [4^ 4 5^ 5] +
-1.0 [5^ 3] +
-25.0 [5^ 5]

Hubbard Hamiltonian in Pauli basis:
(-37.5+0j) [] +
(-0.5+0j) [X0 Z1 X2] +
(-0.5+0j) [Y0 Z1 Y2] +
(12.5+0j) [Z0 Z1] +
(-0.5+0j) [X1 Z2 X3] +
(-0.5+0j) [Y1 Z2 Y3] +
(-0.5+0j) [X2 Z3 X4] +
(-0.5+0j) [Y2 Z3 Y4] +
(12.5+0j) [Z2 Z3] +
(-0.5+0j) [X3 Z4 X5] +
(-0.5+0j) [Y3 Z4 Y5] +
(12.5+0j) [Z4 Z5]
List of terms
{((0, 'Y'), (1, 'Z'), (2, 'Y')): (-0.5+0j), ((0, 'X'), (1, 'Z'), (2, 'X')): (-0.5+0j), ((1, 'Y'), (2, 'Z'), (3, 'Y')): (-0.5+0j), ((1, 'X'), (2, 'Z'), (3, 'X')): (-0.5+0j), ((0, 'Z'), (1, 'Z')): (12.5+0j), ((2, 'Y'), (3, 'Z'), (4, 'Y')): (-0.5+0j), ((2, 'X'), (3, 'Z'), (4, 'X')): (-0.5+0j), ((3, 'Y'), (4, 'Z'), (5, 'Y')): (-0.5+0j), ((3, 'X'), (4, 'Z'), (5, 'X')): (-0.5+0j), ((2, 'Z'), (3, 'Z')): (12.5+0j), (

/tmp/ipykernel_15924/3091458728.py:209: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 2.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for instr, qargs, _ in circuit.data:
/tmp/ipykernel_15924/3091458728.py:218: DeprecationWarning: Treating CircuitInstruction as an iterable is deprecated legacy behavior since Qiskit 1.2, and will be removed in Qiskit 2.0. Instead, use the `operation`, `qubits` and `clbits` named attributes.
  for instr, qargs, cargs in circuit.data:


QiskitError: 'Keyboard interrupt in parallel_map.'